In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.optimize import minimize

In [ ]:
# ============================================================
# Helpers
# ============================================================
def _softmax_util(U):
    """
    U: (N, M) utility matrix
    return: (N, M) probabilities
    """
    U = np.asarray(U, dtype=float)
    Umax = np.max(U, axis=1, keepdims=True)
    ex = np.exp(U - Umax)
    denom = np.sum(ex, axis=1, keepdims=True)
    return ex / denom

def _prepare_od_multimode(
    od_df,
    origin_col,
    dest_col,
    flow_col,
    time_cols
):
    req = {origin_col, dest_col, flow_col, *time_cols.values()}
    missing = req - set(od_df.columns)
    if missing:
        raise ValueError(f"od_df missing columns: {missing}")

    od = od_df.copy()
    od = od.dropna(subset=[origin_col, dest_col, flow_col])
    od = od[od[flow_col] > 0].copy()

    if od.empty:
        raise ValueError("After dropping NAs / nonpositive flows, od_df is empty.")

    denom = od.groupby(origin_col)[flow_col].transform("sum")
    od["w_ij"] = od[flow_col] / denom
    return od

def _prepare_origin_share_multimode(
    p_i_df,
    origin_col,
    share_cols
):
    req = {origin_col, *share_cols.values()}
    missing = req - set(p_i_df.columns)
    if missing:
        raise ValueError(f"p_i_df missing columns: {missing}")

    p = p_i_df[[origin_col, *share_cols.values()]].copy()
    p = p.dropna(subset=[origin_col, *share_cols.values()])

    arr = p[list(share_cols.values())].to_numpy(dtype=float)
    if np.any(arr < 0) or np.any(arr > 1):
        raise ValueError("Observed mode shares must be in [0, 1].")

    #row_sum = arr.sum(axis=1)
    #if not np.allclose(row_sum, 1.0, atol=1e-4):
        #raise ValueError("Observed mode shares must sum to 1 for each origin.")

    return p

def _merge_target_multimode(od, p, origin_col):
    merged = od.merge(p, on=origin_col, how="inner", validate="many_to_one")
    if merged.empty:
        raise ValueError("No overlap between OD origins and p_i origins after merge.")
    return merged

def _aggregate_origin_mode_share(merged, origin_col, prob_cols):
    """
    prob_cols: list of columns like ['P_car_ij', 'P_pt_ij', ...]
    returns DataFrame indexed by origin_col
    """
    out = merged[[origin_col, "w_ij"]].copy()
    for c in prob_cols:
        out[c] = merged["w_ij"].to_numpy() * merged[c].to_numpy()
    agg = out.groupby(origin_col)[prob_cols].sum()
    return agg

def _loss_sse_multimode(obs_df, hat_df, share_order, weight=None):
    """
    obs_df, hat_df indexed by origin_col
    share_order: list of mode names, e.g. ['car','pt','walk','bike']
    """
    resid_list = []
    for m in share_order:
        resid_list.append(
            obs_df[f"p_{m}_true"].to_numpy() - hat_df[f"p_{m}_hat"].reindex(obs_df.index).to_numpy()
        )
    resid = np.column_stack(resid_list)  # (n_origins, n_modes)

    if weight is None:
        return float(np.sum(resid ** 2))

    w = weight.reindex(obs_df.index).fillna(1.0).to_numpy().reshape(-1, 1)
    return float(np.sum(w * resid ** 2))

def calibrate_mnl_time_only(
    od_df,
    p_i_df,
    origin_col="o_id",
    dest_col="d_id",
    flow_col="flow",
    time_cols=None,
    share_cols=None,
    use_origin_weight=False,
    walk_time_cap=None,
    bike_time_cap=None
):
    """
    Multinomial logit with common time coefficient:
        U_car  = asc_car  + beta * t_car
        U_pt   = 0        + beta * t_pt
        U_walk = asc_walk + beta * t_walk
        U_bike = asc_bike + beta * t_bike

    time_cols example:
        {
            "car":  "drive_time",
            "pt":   "pt_time",
            "walk": "walk_time",
            "bike": "bike_time"
        }

    share_cols example:
        {
            "car":  "p_car",
            "pt":   "p_pt",
            "walk": "p_walk",
            "bike": "p_bike"
        }
    """
    if time_cols is None:
        time_cols = {
            "car": "drive_time",
            "pt": "pt_time",
            "walk": "walk_time",
            "bike": "bike_time"
        }

    if share_cols is None:
        share_cols = {
            "car": "p_car",
            "pt": "p_pt",
            "walk": "p_walk",
            "bike": "p_bike"
        }

    modes = ["car", "pt", "walk", "bike"]

    od = _prepare_od_multimode(
        od_df=od_df,
        origin_col=origin_col,
        dest_col=dest_col,
        flow_col=flow_col,
        time_cols=time_cols
    )

    p = _prepare_origin_share_multimode(
        p_i_df=p_i_df,
        origin_col=origin_col,
        share_cols=share_cols
    )

    merged = _merge_target_multimode(od, p, origin_col)

    # observed origin shares
    p_true = merged.groupby(origin_col)[list(share_cols.values())].first()
    p_true.columns = [f"p_{m}_true" for m in modes]

    # optional weights
    origin_weight = None
    if use_origin_weight:
        origin_weight = merged.groupby(origin_col)[flow_col].sum()

    # time arrays
    t_car = merged[time_cols["car"]].to_numpy(dtype=float)
    t_pt = merged[time_cols["pt"]].to_numpy(dtype=float)
    t_walk = merged[time_cols["walk"]].to_numpy(dtype=float)
    t_bike = merged[time_cols["bike"]].to_numpy(dtype=float)

    # availability masks
    avail_car = np.isfinite(t_car)
    avail_pt = np.isfinite(t_pt)
    avail_walk = np.isfinite(t_walk)
    avail_bike = np.isfinite(t_bike)

    if walk_time_cap is not None:
        avail_walk = avail_walk & (t_walk <= walk_time_cap)
    if bike_time_cap is not None:
        avail_bike = avail_bike & (t_bike <= bike_time_cap)

    def _compute_probs(theta):
        asc_car, asc_walk, asc_bike, beta_time = theta

        U = np.full((len(merged), 4), -1e12, dtype=float)

        # car
        U[avail_car, 0] = asc_car + beta_time * t_car[avail_car]
        # pt (base ASC = 0)
        U[avail_pt, 1] = 0.0 + beta_time * t_pt[avail_pt]
        # walk
        U[avail_walk, 2] = asc_walk + beta_time * t_walk[avail_walk]
        # bike
        U[avail_bike, 3] = asc_bike + beta_time * t_bike[avail_bike]

        # if a row has no available mode, raise error
        row_max = np.max(U, axis=1)
        if np.any(row_max < -1e11):
            raise ValueError("Some OD rows have no available modes.")

        P = _softmax_util(U)
        return P

    def objective(theta):
        P = _compute_probs(theta)

        tmp = merged[[origin_col, "w_ij"]].copy()
        tmp["P_car_ij"] = P[:, 0]
        tmp["P_pt_ij"] = P[:, 1]
        tmp["P_walk_ij"] = P[:, 2]
        tmp["P_bike_ij"] = P[:, 3]

        p_hat = _aggregate_origin_mode_share(
            tmp,
            origin_col=origin_col,
            prob_cols=["P_car_ij", "P_pt_ij", "P_walk_ij", "P_bike_ij"]
        )
        p_hat.columns = ["p_car_hat", "p_pt_hat", "p_walk_hat", "p_bike_hat"]

        return _loss_sse_multimode(
            obs_df=p_true,
            hat_df=p_hat,
            share_order=modes,
            weight=origin_weight
        )

    # initial values
    x0 = np.array([0.0, 0.0, 0.0, -0.05], dtype=float)

    # keep beta negative
    bounds = [
        (None, None),   # asc_car
        (None, None),   # asc_walk
        (None, None),   # asc_bike
        (-10.0, -1e-6)  # beta_time
    ]

    res = minimize(objective, x0=x0, method="L-BFGS-B", bounds=bounds)

    theta_hat = res.x
    asc_car_hat, asc_walk_hat, asc_bike_hat, beta_hat = theta_hat

    P_hat = _compute_probs(theta_hat)

    merged_out = merged.copy()
    merged_out["P_car_ij"] = P_hat[:, 0]
    merged_out["P_pt_ij"] = P_hat[:, 1]
    merged_out["P_walk_ij"] = P_hat[:, 2]
    merged_out["P_bike_ij"] = P_hat[:, 3]

    p_hat = _aggregate_origin_mode_share(
        merged_out,
        origin_col=origin_col,
        prob_cols=["P_car_ij", "P_pt_ij", "P_walk_ij", "P_bike_ij"]
    )
    p_hat.columns = ["p_car_hat", "p_pt_hat", "p_walk_hat", "p_bike_hat"]

    diag = p_true.merge(p_hat, left_index=True, right_index=True, how="left").reset_index()

    for m in modes:
        diag[f"resid_{m}"] = diag[f"p_{m}_true"] - diag[f"p_{m}_hat"]

    resid_cols = [f"resid_{m}" for m in modes]
    rmse = float(np.sqrt(np.mean(diag[resid_cols].to_numpy() ** 2)))

    info = {
        "model": "mnl_time_only_common_beta",
        "asc_car": float(asc_car_hat),
        "asc_pt": 0.0,
        "asc_walk": float(asc_walk_hat),
        "asc_bike": float(asc_bike_hat),
        "beta_time": float(beta_hat),
        "success": bool(res.success),
        "message": res.message,
        "loss_sse": float(res.fun),
        "rmse_origin_share_all_modes": rmse,
        "n_od": int(len(merged_out)),
        "n_origins": int(diag.shape[0])
    }

    return merged_out, diag, info

In [ ]:
city = 'london'
unit_name = 'msoa'
od_df = pd.read_csv(f'D:/urban_hierarchy_congestion/data/od_tables/{city}_od_time.csv')
od_df = od_df.loc[(od_df['o_id']!=od_df['d_id'])&(od_df['flow']>0)].copy()
od_df.index = range(len(od_df))
p_i_df = od_df.groupby('o_id',as_index=False)['flow'].sum()
unit = gpd.read_file(f"D:/urban_hierarchy_congestion/data/taz/{city}_{unit_name}.shp")
p_i_df = pd.merge(p_i_df,unit[['id','p_car','p_pt','p_bike','p_walk']],left_on='o_id',right_on='id')
p_i_df.drop(columns=['id','flow'],inplace=True)

time_cols = {
    "car": "drive_time",
    "pt": "pt_time",
    "walk": "walk_time",
    "bike": "bike_time"
}

share_cols = {
    "car": "p_car",
    "pt": "p_pt",
    "walk": "p_walk",
    "bike": "p_bike"
}

merged_out, diag, info = calibrate_mnl_time_only(
    od_df=od_df,
    p_i_df=p_i_df,
    origin_col="o_id",
    dest_col="d_id",
    flow_col="flow",
    time_cols=time_cols,
    share_cols=share_cols,
    use_origin_weight=True,
    walk_time_cap=90,   # Can be adjusted as desired
    bike_time_cap=60    # Can be adjusted as desired
)

print(info)

In [ ]:
# ============================================================
# Basic helpers
# ============================================================
def softmax_util(U):
    """
    U: (N, M) utility matrix
    return: (N, M) probabilities
    """
    U = np.asarray(U, dtype=float)
    Umax = np.max(U, axis=1, keepdims=True)
    ex = np.exp(U - Umax)
    denom = np.sum(ex, axis=1, keepdims=True)
    return ex / denom


def compute_mode_probabilities(
    car_time,
    pt_time,
    walk_time,
    bike_time,
    asc_car,
    asc_walk,
    asc_bike,
    beta,
    walk_time_cap=90,
    bike_time_cap=60
):
    """
    PT is the reference alternative:
        U_car  = asc_car  + beta * t_car
        U_pt   = 0        + beta * t_pt
        U_walk = asc_walk + beta * t_walk
        U_bike = asc_bike + beta * t_bike
    """

    car_time = np.asarray(car_time, dtype=float)
    pt_time = np.asarray(pt_time, dtype=float)
    walk_time = np.asarray(walk_time, dtype=float)
    bike_time = np.asarray(bike_time, dtype=float)

    n = len(car_time)
    U = np.full((n, 4), -1e12, dtype=float)

    avail_car = np.isfinite(car_time) & (car_time > 0)
    avail_pt = np.isfinite(pt_time) & (pt_time > 0)
    avail_walk = np.isfinite(walk_time) & (walk_time > 0)
    avail_bike = np.isfinite(bike_time) & (bike_time > 0)

    if walk_time_cap is not None:
        avail_walk = avail_walk & (walk_time <= walk_time_cap)
    if bike_time_cap is not None:
        avail_bike = avail_bike & (bike_time <= bike_time_cap)

    U[avail_car, 0] = asc_car + beta * car_time[avail_car]
    U[avail_pt, 1] = 0.0 + beta * pt_time[avail_pt]
    U[avail_walk, 2] = asc_walk + beta * walk_time[avail_walk]
    U[avail_bike, 3] = asc_bike + beta * bike_time[avail_bike]

    if np.any(np.max(U, axis=1) < -1e11):
        raise ValueError("Some OD pairs have no available modes after filtering.")

    P = softmax_util(U)

    return {
        "p_car": P[:, 0],
        "p_pt": P[:, 1],
        "p_walk": P[:, 2],
        "p_bike": P[:, 3]
    }


def weighted_predicted_mode_shares(
    asc,
    car_time,
    pt_time,
    walk_time,
    bike_time,
    weights,
    beta,
    walk_time_cap=90,
    bike_time_cap=60
):
    """
    asc = [asc_car, asc_walk, asc_bike]
    returns weighted city-wide predicted shares
    """
    asc_car, asc_walk, asc_bike = asc

    probs = compute_mode_probabilities(
        car_time=car_time,
        pt_time=pt_time,
        walk_time=walk_time,
        bike_time=bike_time,
        asc_car=asc_car,
        asc_walk=asc_walk,
        asc_bike=asc_bike,
        beta=beta,
        walk_time_cap=walk_time_cap,
        bike_time_cap=bike_time_cap
    )

    w = np.asarray(weights, dtype=float)
    wsum = np.sum(w)

    if wsum <= 0:
        raise ValueError("Sum of weights must be positive.")

    return {
        "car": float(np.sum(w * probs["p_car"]) / wsum),
        "pt": float(np.sum(w * probs["p_pt"]) / wsum),
        "walk": float(np.sum(w * probs["p_walk"]) / wsum),
        "bike": float(np.sum(w * probs["p_bike"]) / wsum)
    }


# ============================================================
# Calibrate one city
# ============================================================
def calibrate_asc_for_one_city(
    car_time,
    pt_time,
    walk_time,
    bike_time,
    weights,
    target_share,
    beta,
    walk_time_cap=90,
    bike_time_cap=60,
    x0=(0.0, -1.0, -0.5),
    method="L-BFGS-B"
):
    """
    Calibrate asc_car, asc_walk, asc_bike for one city
    given fixed beta and target city-wide mode shares.

    target_share example:
        {"car": 0.35, "pt": 0.40, "walk": 0.10, "bike": 0.15}
    """

    car_time = np.asarray(car_time, dtype=float)
    pt_time = np.asarray(pt_time, dtype=float)
    walk_time = np.asarray(walk_time, dtype=float)
    bike_time = np.asarray(bike_time, dtype=float)
    weights = np.asarray(weights, dtype=float)

    valid = (
        np.isfinite(weights) &
        (weights > 0) &
        (
            (np.isfinite(car_time) & (car_time > 0)) |
            (np.isfinite(pt_time) & (pt_time > 0)) |
            (np.isfinite(walk_time) & (walk_time > 0)) |
            (np.isfinite(bike_time) & (bike_time > 0))
        )
    )

    car_time = car_time[valid]
    pt_time = pt_time[valid]
    walk_time = walk_time[valid]
    bike_time = bike_time[valid]
    weights = weights[valid]

    if len(car_time) == 0:
        raise ValueError("No valid OD pairs after filtering.")

    modes = ["car", "pt", "walk", "bike"]
    missing_modes = set(modes) - set(target_share.keys())
    if missing_modes:
        raise ValueError(f"target_share missing modes: {missing_modes}")

    tgt = np.array([target_share[m] for m in modes], dtype=float)

    if np.any(tgt < 0) or np.any(tgt > 1):
        raise ValueError("All target shares must be in [0, 1].")
    if not np.isclose(np.sum(tgt), 1.0, atol=1e-8):
        raise ValueError("Target shares must sum to 1.")

    def objective(theta):
        pred = weighted_predicted_mode_shares(
            asc=theta,
            car_time=car_time,
            pt_time=pt_time,
            walk_time=walk_time,
            bike_time=bike_time,
            weights=weights,
            beta=beta,
            walk_time_cap=walk_time_cap,
            bike_time_cap=bike_time_cap
        )
        pred_vec = np.array([pred[m] for m in modes], dtype=float)
        resid = pred_vec - tgt
        return float(np.sum(resid ** 2))

    res = minimize(
        objective,
        x0=np.array(x0, dtype=float),
        method=method
    )

    asc_hat = res.x
    pred = weighted_predicted_mode_shares(
        asc=asc_hat,
        car_time=car_time,
        pt_time=pt_time,
        walk_time=walk_time,
        bike_time=bike_time,
        weights=weights,
        beta=beta,
        walk_time_cap=walk_time_cap,
        bike_time_cap=bike_time_cap
    )

    out = {
        "asc_car": float(asc_hat[0]),
        "asc_pt": 0.0,
        "asc_walk": float(asc_hat[1]),
        "asc_bike": float(asc_hat[2]),
        "pred_car": pred["car"],
        "pred_pt": pred["pt"],
        "pred_walk": pred["walk"],
        "pred_bike": pred["bike"],
        "success": bool(res.success),
        "message": res.message,
        "loss_sse": float(res.fun),
        "valid_od_n": int(len(car_time))
    }
    return out


# ============================================================
# Calibrate all cities
# ============================================================
def calibrate_asc_by_city(
    df,
    city_col,
    car_time_col,
    pt_time_col,
    walk_time_col,
    bike_time_col,
    weight_col,
    target_share_map,
    beta,
    walk_time_cap=90,
    bike_time_cap=60,
    x0=(0.0, -1.0, -0.5)
):
    """
    target_share_map example:
    {
        "beijing":  {"car": 0.35, "pt": 0.40, "walk": 0.10, "bike": 0.15},
        "shanghai": {"car": 0.35, "pt": 0.40, "walk": 0.10, "bike": 0.15},
        ...
    }
    """

    results = []

    for city, target_share in target_share_map.items():
        sub = df[df[city_col] == city].copy()

        if sub.empty:
            raise ValueError(f"No rows found for city: {city}")

        fit = calibrate_asc_for_one_city(
            car_time=sub[car_time_col].values,
            pt_time=sub[pt_time_col].values,
            walk_time=sub[walk_time_col].values,
            bike_time=sub[bike_time_col].values,
            weights=sub[weight_col].values,
            target_share=target_share,
            beta=beta,
            walk_time_cap=walk_time_cap,
            bike_time_cap=bike_time_cap,
            x0=x0
        )

        results.append({
            "city": city,
            "beta": beta,
            "asc_car": fit["asc_car"],
            "asc_pt": fit["asc_pt"],
            "asc_walk": fit["asc_walk"],
            "asc_bike": fit["asc_bike"],
            "target_car": target_share["car"],
            "target_pt": target_share["pt"],
            "target_walk": target_share["walk"],
            "target_bike": target_share["bike"],
            "pred_car": fit["pred_car"],
            "pred_pt": fit["pred_pt"],
            "pred_walk": fit["pred_walk"],
            "pred_bike": fit["pred_bike"],
            "abs_err_car": abs(fit["pred_car"] - target_share["car"]),
            "abs_err_pt": abs(fit["pred_pt"] - target_share["pt"]),
            "abs_err_walk": abs(fit["pred_walk"] - target_share["walk"]),
            "abs_err_bike": abs(fit["pred_bike"] - target_share["bike"]),
            "loss_sse": fit["loss_sse"],
            "success": fit["success"],
            "message": fit["message"],
            "valid_od_n": fit["valid_od_n"]
        })

    return pd.DataFrame(results)

In [ ]:
od_all = pd.DataFrame()
for city in ['beijing','shanghai','shenzhen']:
    od_df = pd.read_csv(f'D:/urban_hierarchy_congestion/data/od_tables/{city}_od_time.csv')
    od_df = od_df.loc[(od_df['o_id']!=od_df['d_id'])&(od_df['flow']>0)].copy()
    od_df.index = range(len(od_df))
    od_df['city'] = city
    od_all = pd.concat([od_all,od_df],axis=0,ignore_index=True)

In [ ]:
target_share_map = {
    "beijing":  {"car": 0.35, "pt": 0.40, "walk": 0.10, "bike": 0.15},
    "shanghai": {"car": 0.35, "pt": 0.40, "walk": 0.10, "bike": 0.15},
    "shenzhen": {"car": 0.35, "pt": 0.40, "walk": 0.10, "bike": 0.15},
    "nanjing": {"car": 0.35, "pt": 0.40, "walk": 0.10, "bike": 0.15}
}

result_df = calibrate_asc_by_city(
    df=od_all,
    city_col="city",
    car_time_col="drive_time",
    pt_time_col="pt_time",
    walk_time_col="walk_time",
    bike_time_col="bike_time",
    weight_col="flow",
    target_share_map=target_share_map,
    beta=-0.128,
    walk_time_cap=90,
    bike_time_cap=60,
    x0=(0.0, -1.5, -0.8)
)

asc_map = {
    row["city"]: {
        "asc_car": row["asc_car"],
        "asc_walk": row["asc_walk"],
        "asc_bike": row["asc_bike"]
    }
    for _, row in result_df.iterrows()
}

print(asc_map)